In [7]:
from pyecharts import options as opts
from pyecharts.charts import Tree
import pandas as pd

In [8]:
df = pd.read_excel("../data/AMDP_en.xlsx", dtype=str)
df = df.apply(lambda s: s.str.strip()).dropna(how="all")
df.columns = ["Level1", "Level2","Level3", "Level4","Ref"]
df = df.drop("Ref", axis = 1)
df

,Level1,Level2,Level3,Level4
0,Psychopathological Symptoms,Disorders of Conscousness,Lowered Vigilance,NaN
1,Psychopathological Symptoms,Disorders of Conscousness,Clouded Consciousness,NaN
2,Psychopathological Symptoms,Disorders of Conscousness,Narrowed Consciousness,NaN
3,Psychopathological Symptoms,Disorders of Conscousness,Expanded Consciousness,NaN
4,Psychopathological Symptoms,Disturbances of Orientation,Time,NaN
...,...,...,...,...
142,Somatic Signs,Neurological Disturbances,Hypokinesia,NaN
143,Somatic Signs,Neurological Disturbances,Akathisia,NaN
144,Somatic Signs,Neurological Disturbances,Ataxia,NaN
145,Somatic Signs,Neurological Disturbances,Nystagmus,NaN


In [9]:
arbre_dict = {}
for chemin in df.itertuples(index=False):
    noeud = arbre_dict
    for nom in chemin:
        if pd.isna(nom):          # niveau absent : on arrête ce chemin
            break
        noeud = noeud.setdefault(nom, {})

def convertir(d):
    return [{"name": k, "children": convertir(v)} for k, v in d.items()]

racine = {"name": "AMDP", "children": convertir(arbre_dict)}


COULEURS = {
    "Psychopathological Symptoms": "#65C5CA",
    "Somatic Signs": "#f7cf71",
}
DEFAUT = "#CCCCCC"
PAS = 0.3   # éclaircissement 

def eclaircir(hex_c, f):
    h = hex_c.lstrip("#")
    rgb = [int(h[i:i+2], 16) for i in (0, 2, 4)]
    return "#%02x%02x%02x" % tuple(round(v + (255 - v) * f) for v in rgb)

def colorier(noeud, c, prof=0):
    cc = eclaircir(c, min(PAS * prof, 0.7))
    noeud["itemStyle"] = {"color": cc}
    noeud["lineStyle"] = {"color": cc, "width": 2}
    for e in noeud.get("children", []):
        colorier(e, c, prof + 1)

for branche in racine["children"]:
    colorier(branche, COULEURS.get(branche["name"], DEFAUT))

for branche in racine["children"]:
    branche["lineStyle"] = {**branche["lineStyle"], "curveness": 0}

arbre = Tree(init_opts=opts.InitOpts(width="1200px", height="1200px")).add("", [racine], layout="radial",initial_tree_depth=-1,label_opts=opts.LabelOpts(font_size=10))
arbre.render("../html/tree_AMDP.html")
arbre.render_notebook()